# 02 — Retrieval Pipeline
**PaperSloth — Hybrid Search + Rerank + Parent Expansion + Gemini**

```
Query
  → Redis cache check
  → Embed query (Ollama)
  → BM25 encode query
  → Pinecone hybrid search + metadata filter
  → Cross-encoder rerank (local)
  → Expand child → parent (PostgreSQL)
  → Gemini generation
  → Cache result (Redis)
```

---
## Section 1 — Setup & Connections

In [1]:
import os, json, pickle, hashlib, time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

keys = ["GEMINI_API_KEY", "PINECONE_API_KEY", "DATABASE_URL"]
for k in keys:
    print(f"{k}: {'✅' if os.getenv(k) else '❌ MISSING'}")

GEMINI_API_KEY: ✅
PINECONE_API_KEY: ✅
DATABASE_URL: ✅


In [3]:
# Pinecone
from pinecone import Pinecone
pc    = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("papersloth")
stats = index.describe_index_stats()
print(f"✅ Pinecone — {stats['total_vector_count']} vectors")

✅ Pinecone — 16 vectors


In [4]:
# PostgreSQL
import psycopg2
conn_test = psycopg2.connect(os.getenv("DATABASE_URL"))
cur       = conn_test.cursor()
cur.execute("SELECT COUNT(*) FROM parent_chunks;")
count = cur.fetchone()[0]
cur.close(); conn_test.close()
print(f"✅ PostgreSQL — {count} parent chunks")

✅ PostgreSQL — 4 parent chunks


In [7]:
# Redis — optional, graceful fallback if not running
import redis as redis_lib
try:
    r = redis_lib.from_url(os.getenv("REDIS_URL", "redis://localhost:6379"))
    r.ping()
    REDIS_ON = True
    print("✅ Redis connected")
except Exception as e:
    REDIS_ON = False
    print(f"⚠️  Redis not available — caching disabled ({e})")
    print("   To enable: brew install redis && brew services start redis")

✅ Redis connected


In [21]:
# Gemini
import google.generativeai as genai
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemma-4-31b-it")
print("✅ Gemini ready")

✅ Gemini ready


In [9]:
# Ollama embedding model
import ollama
EMBED_MODEL = "nomic-embed-text-v2-moe:latest"
test = ollama.embed(model=EMBED_MODEL, input="search_query: test")
print(f"✅ Ollama embed — dim: {len(test['embeddings'][0])}")

✅ Ollama embed — dim: 768


In [10]:
# Load BM25 model saved during ingestion
BM25_PATH = "data/bm25_model.pkl"
assert Path(BM25_PATH).exists(), f"BM25 model not found at {BM25_PATH}. Run 01_ingestion first."
with open(BM25_PATH, "rb") as f:
    bm25 = pickle.load(f)
print("✅ BM25 model loaded")

✅ BM25 model loaded


In [11]:
# Cross-encoder reranker — runs locally on CPU, ~80MB download on first run
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Reranker loaded")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Reranker loaded


---
## Section 2 — Helper Functions

We build each component as a small testable function before combining them.

In [12]:
# ── 2.1 Query embedding ──────────────────────────────────────────────────────
# nomic-embed-text requires:
#   'search_document: ...' prefix when embedding documents (ingestion)
#   'search_query: ...'    prefix when embedding queries   (retrieval)

def embed_query(text: str) -> list[float]:
    resp = ollama.embed(model=EMBED_MODEL, input=f"search_query: {text}")
    return resp["embeddings"][0]

# Quick test
v = embed_query("thermocouple temperature measurement")
print(f"Query vector dim: {len(v)}  first3: {v[:3]}")

Query vector dim: 768  first3: [0.004199332, -0.010849649, -0.009699033]


In [13]:
# ── 2.2 Hybrid score normalisation ──────────────────────────────────────────
# alpha controls the dense/sparse balance:
#   1.0 → pure semantic (dense only)
#   0.0 → pure keyword  (sparse/BM25 only)
#   0.7 → 70% semantic + 30% keyword  ← recommended for exam questions
#
# Why 0.7? Exam questions have important domain keywords ("Dijkstra",
# "thermocouple", "PI controller"). We want keyword matching but still
# benefit from semantic similarity.

def hybrid_scale(dense: list[float], sparse: dict, alpha: float = 0.7):
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be 0–1")
    scaled_sparse = {
        "indices": sparse["indices"],
        "values":  [v * (1 - alpha) for v in sparse["values"]],
    }
    scaled_dense = [v * alpha for v in dense]
    return scaled_dense, scaled_sparse

print("✅ hybrid_scale defined")

✅ hybrid_scale defined


In [14]:
# ── 2.3 Redis cache helpers ──────────────────────────────────────────────────
# We hash (query + filters) → unique cache key
# TTL = 24 hours. When new papers are ingested, flush the cache manually.

CACHE_TTL = 60 * 60 * 24  # 24 hours in seconds

def _cache_key(query: str, filters: dict) -> str:
    raw = f"{query.strip().lower()}:{json.dumps(filters, sort_keys=True)}"
    return f"ps:{hashlib.md5(raw.encode()).hexdigest()}"

def cache_get(query: str, filters: dict):
    if not REDIS_ON:
        return None
    val = r.get(_cache_key(query, filters))
    return json.loads(val) if val else None

def cache_set(query: str, filters: dict, result: dict):
    if not REDIS_ON:
        return
    r.setex(_cache_key(query, filters), CACHE_TTL, json.dumps(result))

print("✅ Cache helpers defined")

✅ Cache helpers defined


In [15]:
# ── 2.4 PostgreSQL parent fetcher ────────────────────────────────────────────
# After Pinecone returns child IDs, we look up the full parent document.
# This is the "parent expansion" step of MultiVector Retriever.

def fetch_parents(parent_ids: list[str]) -> list[dict]:
    if not parent_ids:
        return []
    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    cur  = conn.cursor()
    cur.execute("""
        SELECT parent_id, question_number, full_text, total_marks,
               image_urls, course_code, semester, year
        FROM   parent_chunks
        WHERE  parent_id = ANY(%s)
    """, (parent_ids,))
    rows = cur.fetchall()
    cur.close(); conn.close()
    return [
        {
            "parent_id":       row[0],
            "question_number": row[1],
            "full_text":       row[2],
            "total_marks":     row[3],
            "image_urls":      row[4] or {},
            "course_code":     row[5],
            "semester":        row[6],
            "year":            row[7],
        }
        for row in rows
    ]

print("✅ fetch_parents defined")

✅ fetch_parents defined


---
## Section 3 — Hybrid Search

Test the search before adding reranking and generation on top.

In [16]:
# ── 3.1  Build metadata filter ───────────────────────────────────────────────
# Metadata filter runs BEFORE vector search inside Pinecone.
# This narrows the search to only matching documents — faster and more precise.
#
# Leave a field out to not filter on it.
# Example filters:
#   {"course_code": {"$eq": "RBB3013"}}                 ← specific subject
#   {"year": {"$gte": 2022}}                              ← recent papers only
#   {"question_type": {"$eq": "calculation"}}             ← only calc questions
#   {"course_code": {"$eq": "RBB3013"}, "year": {"$eq": 2025}}  ← combined

def build_filter(
    course_code:    str  = None,
    year:           int  = None,
    semester:       str  = None,
    question_type:  str  = None,
    min_marks:      int  = None,
) -> dict:
    f = {}
    if course_code:   f["course_code"]   = {"$eq": course_code}
    if year:          f["year"]           = {"$eq": year}
    if semester:      f["semester"]       = {"$eq": semester}
    if question_type: f["question_type"]  = {"$eq": question_type}
    if min_marks:     f["marks"]          = {"$gte": min_marks}
    return f

print("✅ build_filter defined")
print("Example:", build_filter(course_code="RBB3013", question_type="calculation"))

✅ build_filter defined
Example: {'course_code': {'$eq': 'RBB3013'}, 'question_type': {'$eq': 'calculation'}}


In [17]:
# ── 3.2  Run hybrid search ───────────────────────────────────────────────────

TEST_QUERY = "thermocouple temperature voltage measurement"   # ← change to test
TEST_FILTER = build_filter(course_code="RBB3013")            # ← or {} for no filter

t0 = time.time()

# Embed query
dense  = embed_query(TEST_QUERY)
sparse = bm25.encode_queries([TEST_QUERY])[0]

# Scale for hybrid
d_scaled, s_scaled = hybrid_scale(dense, sparse, alpha=0.7)

# Search Pinecone
results = index.query(
    vector        = d_scaled,
    sparse_vector = s_scaled,
    top_k         = 20,
    include_metadata = True,
    filter        = TEST_FILTER if TEST_FILTER else None,
)

elapsed = time.time() - t0
print(f"✅ Hybrid search returned {len(results.matches)} results in {elapsed:.2f}s")
print()
for m in results.matches:
    print(f"  score={m.score:.4f}  "
          f"Q{m.metadata['question_number']}{m.metadata.get('sub_question','')}  "
          f"{m.metadata['course_code']} {m.metadata['semester']}  "
          f"[{m.metadata.get('marks',0)}m]  {m.metadata.get('question_type','')}")
    print(f"         {m.metadata.get('text_preview','')[:80]}...")

✅ Hybrid search returned 16 results in 1.50s

  score=0.6499  Q2c.i  RBB3013 September 2025  [5m]  calculation
         Thermocouple is one of the most used techniques to measure temperature in the oi...
  score=0.6147  Q2c.ii  RBB3013 September 2025  [5m]  calculation
         Thermocouple is one of the most used techniques to measure temperature in the oi...
  score=0.3184  Q1a.i  RBB3013 September 2025  [4m]  calculation
         The output voltage from a precision 12-V power supply is monitored at intervals ...
  score=0.2931  Q1a.ii  RBB3013 September 2025  [3m]  theory
         The output voltage from a precision 12-V power supply is monitored at intervals ...
  score=0.2842  Q1a.iii  RBB3013 September 2025  [3m]  theory
         The output voltage from a precision 12-V power supply is monitored at intervals ...
  score=0.2423  Q1b.i  RBB3013 September 2025  [10m]  calculation
         A multirange voltmeter is constructed using a permanent magnet moving coil (PMMC...
  score=0.2

---
## Section 4 — Reranking

Pinecone returns up to 20 candidates based on vector similarity.
The cross-encoder reranker reads the query AND each passage together — much more accurate than embedding similarity alone.
We keep only the top N after reranking.

In [18]:
# ── 4.1  Rerank ──────────────────────────────────────────────────────────────
# Cross-encoder scores (query, passage) pairs.
# Higher score = more relevant.

RERANK_TOP_N = 5   # how many to keep after reranking

def rerank(query: str, matches: list, top_n: int = RERANK_TOP_N) -> list:
    """
    matches: list of Pinecone match objects
    Returns top_n matches sorted by cross-encoder score (best first)
    """
    if not matches:
        return []

    # Use text_preview stored in metadata — good enough for reranking
    passages = [m.metadata.get("text_preview", "") for m in matches]
    pairs    = [(query, p) for p in passages]

    scores   = reranker.predict(pairs)
    ranked   = sorted(
        zip(matches, scores),
        key=lambda x: x[1],
        reverse=True
    )
    return [match for match, _ in ranked[:top_n]]


# Test on previous search results
t0 = time.time()
top_matches = rerank(TEST_QUERY, results.matches)
elapsed = time.time() - t0

print(f"✅ Reranked to top {len(top_matches)} in {elapsed:.2f}s")
print()
for i, m in enumerate(top_matches):
    print(f"  #{i+1}  Q{m.metadata['question_number']}{m.metadata.get('sub_question','')}  "
          f"{m.metadata['course_code']} {m.metadata['semester']}")
    print(f"       {m.metadata.get('text_preview','')[:100]}...")

✅ Reranked to top 5 in 2.11s

  #1  Q2c.ii  RBB3013 September 2025
       Thermocouple is one of the most used techniques to measure temperature in the oil and gas industries...
  #2  Q2c.i  RBB3013 September 2025
       Thermocouple is one of the most used techniques to measure temperature in the oil and gas industries...
  #3  Q1a.i  RBB3013 September 2025
       The output voltage from a precision 12-V power supply is monitored at intervals over a period, which...
  #4  Q1a.ii  RBB3013 September 2025
       The output voltage from a precision 12-V power supply is monitored at intervals over a period, which...
  #5  Q1a.iii  RBB3013 September 2025
       The output voltage from a precision 12-V power supply is monitored at intervals over a period, which...


---
## Section 5 — Parent Expansion

Pinecone returns child chunk IDs.  
We map each child → its parent_id → fetch full parent from PostgreSQL.  
The parent contains the complete question with all sub-parts and image URLs.

In [19]:
# ── 5.1  Expand children → parents ──────────────────────────────────────────

# Collect unique parent IDs from top matches
parent_ids = list({m.metadata["parent_id"] for m in top_matches})
print(f"Unique parent IDs: {parent_ids}")

parents = fetch_parents(parent_ids)
print(f"\nFetched {len(parents)} parent documents")

for p in parents:
    print(f"\n{'='*60}")
    print(f"SOURCE: {p['course_code']} | {p['semester']} {p['year']} | Q{p['question_number']} | {p['total_marks']}m total")
    print(f"IMAGES: {list(p['image_urls'].keys()) if p['image_urls'] else 'none'}")
    print(f"TEXT:\n{p['full_text'][:400]}...")

Unique parent IDs: ['RBB3013_2025_September2025__Q2', 'RBB3013_2025_September2025__Q1']

Fetched 2 parent documents

SOURCE: RBB3013 | September 2025 2025 | Q1 | 25m total
IMAGES: ['TABLE Q1', 'FIGURE Q1b']
TEXT:
Q1a.i: The output voltage from a precision 12-V power supply is monitored at intervals over a period, which produces the set of data shown in TABLE Q1.

| Measurement | Unit (V) |
|---|---|
| V1 | 12.001 |
| V2 | 11.999 |
| V3 | 11.998 |
| V4 | 12.003 |
| V5 | 12.002 |
| V6 | 11.997 |
| V7 | 12.002 |
| V8 | 12.003 |
| V9 | 11.998 |
| V10 | 11.997 |

Evaluate the average value and standard deviatio...

SOURCE: RBB3013 | September 2025 2025 | Q2 | 25m total
IMAGES: ['FIGURE Q2a']
TEXT:
Q2a: FIGURE Q2a shows a pressured vessel level measurement. Solve the Lower Range Value (LRV) and Upper Range Value (URV).

Notes: h1 = 10 ft, h2= 12 ft, d = 5 ft, γ1 = 64 lb/ft³, γ2 = 58.3 lb/ft³ [10 marks]

Q2b: Justify with an example on the requirement of wet leg for a level measurement shown 

---
## Section 6 — Gemini Generation

We pass the full parent documents as context and ask Gemini to answer the student's query.

In [22]:
# ── 6.1  Build context + generate ───────────────────────────────────────────

SYSTEM_PROMPT = """You are PaperSloth, a helpful study assistant for UTP (Universiti Teknologi PETRONAS) students.
You help students find and understand past year exam questions.

When answering:
- Present questions clearly with their question number and marks
- Always state which semester/year the question is from
- If asked for questions on a topic, list them all found in context
- If no relevant questions are found, say so clearly — do not make up questions
- Keep responses concise and focused on what the student asked"""

def generate_answer(query: str, parent_docs: list[dict]) -> str:
    if not parent_docs:
        return "No relevant questions found for your query."

    context_parts = []
    for doc in parent_docs:
        header = (
            f"[{doc['course_code']} | {doc['semester']} {doc['year']} | "
            f"Q{doc['question_number']} | {doc['total_marks']} marks]"
        )
        context_parts.append(f"{header}\n{doc['full_text']}")

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""{SYSTEM_PROMPT}

PAST YEAR QUESTIONS (context):
{context}

STUDENT QUERY: {query}

Answer:"""

    response = model.generate_content(prompt)
    return response.text


# Test
answer = generate_answer(TEST_QUERY, parents)
print(answer)

*   Role: PaperSloth (UTP study assistant).
    *   Goal: Help students find and understand past year exam questions.
    *   Constraint 1: Clear presentation (question number, marks).
    *   Constraint 2: State semester/year.
    *   Constraint 3: List all questions found on the topic.
    *   Constraint 4: No fabrication if not found.
    *   Constraint 5: Concise and focused.
    *   Student Query: "thermocouple temperature voltage measurement"

    *   Scanning the context for "thermocouple", "temperature", or "voltage measurement" (specifically related to thermocouples).
    *   Found:
        *   [RBB3013 | September 2025 2025 | Q2c.i]: Mentions thermocouple, type J, measuring 183°C, reference 25°C, solve voltage in millivolts. [5 marks]
        *   [RBB3013 | September 2025 2025 | Q2c.ii]: Mentions thermocouple, changing from type K to type T, solve voltage in millivolts. [5 marks]

    *   Course/Semester: RBB3013 | September 2025
    *   Q2c.i: Thermocouple type J, 183°C, ref

---
## Section 7 — Full Pipeline

All components combined into one function.

In [23]:
def retrieve_and_generate(
    query:        str,
    course_code:  str  = None,
    year:         int  = None,
    semester:     str  = None,
    question_type:str  = None,
    min_marks:    int  = None,
    top_k:        int  = 20,
    rerank_top_n: int  = 5,
    alpha:        float = 0.7,
) -> dict:
    """
    Full retrieval pipeline.

    Args:
        query:         Natural language question from the student
        course_code:   Filter by subject e.g. 'RBB3013'
        year:          Filter by year e.g. 2025
        semester:      Filter by semester e.g. 'September 2025'
        question_type: Filter by type: 'calculation' | 'theory' | 'diagram' | 'table'
        min_marks:     Only return questions worth at least this many marks
        top_k:         Number of candidates to retrieve from Pinecone
        rerank_top_n:  Number to keep after reranking
        alpha:         Hybrid balance: 1.0=dense only, 0.0=sparse only

    Returns:
        dict with keys: answer, sources, query, filters, cached
    """

    filters = build_filter(course_code, year, semester, question_type, min_marks)

    # ── Step 1: Cache check ─────────────────────────────────────────────────
    cached = cache_get(query, filters)
    if cached:
        cached["cached"] = True
        print(f"⚡ Cache hit")
        return cached

    t_start = time.time()

    # ── Step 2: Embed query ─────────────────────────────────────────────────
    dense  = embed_query(query)
    sparse = bm25.encode_queries([query])[0]
    print(f"  [1/5] embedded ({time.time()-t_start:.2f}s)")

    # ── Step 3: Hybrid search + metadata filter ─────────────────────────────
    d_scaled, s_scaled = hybrid_scale(dense, sparse, alpha)
    results = index.query(
        vector           = d_scaled,
        sparse_vector    = s_scaled,
        top_k            = top_k,
        include_metadata = True,
        filter           = filters if filters else None,
    )
    print(f"  [2/5] hybrid search — {len(results.matches)} candidates ({time.time()-t_start:.2f}s)")

    if not results.matches:
        return {"answer": "No relevant questions found.", "sources": [], "cached": False}

    # ── Step 4: Rerank ──────────────────────────────────────────────────────
    top_matches = rerank(query, results.matches, top_n=rerank_top_n)
    print(f"  [3/5] reranked to top {len(top_matches)} ({time.time()-t_start:.2f}s)")

    # ── Step 5: Parent expansion ────────────────────────────────────────────
    parent_ids = list({m.metadata["parent_id"] for m in top_matches})
    parents    = fetch_parents(parent_ids)
    print(f"  [4/5] fetched {len(parents)} parents ({time.time()-t_start:.2f}s)")

    # ── Step 6: Generate ────────────────────────────────────────────────────
    answer = generate_answer(query, parents)
    print(f"  [5/5] generated ({time.time()-t_start:.2f}s)")

    result = {
        "answer":  answer,
        "sources": parents,
        "query":   query,
        "filters": filters,
        "cached":  False,
    }

    # ── Step 7: Cache ───────────────────────────────────────────────────────
    cache_set(query, filters, result)

    print(f"\nTotal: {time.time()-t_start:.2f}s")
    return result


print("✅ retrieve_and_generate ready")

✅ retrieve_and_generate ready


---
## Section 8 — Test Queries

Test all the different query types we planned.

In [24]:
# ── Test 1: Topic search ─────────────────────────────────────────────────────
result = retrieve_and_generate(
    query       = "thermocouple temperature voltage measurement",
    course_code = "RBB3013",
)
print("\n" + "="*60)
print(result["answer"])

  [1/5] embedded (0.23s)
  [2/5] hybrid search — 16 candidates (1.30s)
  [3/5] reranked to top 5 (1.64s)
  [4/5] fetched 2 parents (3.62s)
  [5/5] generated (17.77s)

Total: 17.78s

PaperSloth (UTP study assistant).
Help students find and understand past year exam questions.
"thermocouple temperature voltage measurement"

        *   Clear question numbers and marks.
        *   State semester/year.
        *   List all relevant questions found in context.
        *   If none found, say so clearly (no hallucinating).
        *   Concise and focused.

    *   Context contains two main questions: Q1 and Q2.
    *   Q1 is about power supply voltage, error analysis, and multirange voltmeters. (Not relevant).
    *   Q2 contains:
        *   Q2a: Pressure vessel level measurement. (Not relevant).
        *   Q2b: Wet leg requirement. (Not relevant).
        *   Q2c.i: "Thermocouple... solve the voltage in millivolts... measures 183°C." (Relevant).
        *   Q2c.ii: "Thermocouple... Solve 

/var/folders/my/2fkdjjvj7h9dn3nmk1ttlccc0000gn/T/ipykernel_19889/1129419027.py:20: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(_cache_key(query, filters), CACHE_TTL, json.dumps(result))


In [25]:
# ── Test 2: Only calculation questions ───────────────────────────────────────
result = retrieve_and_generate(
    query         = "pressure measurement calculate LRV URV",
    course_code   = "RBB3013",
    question_type = "calculation",
)
print("\n" + "="*60)
print(result["answer"])

  [1/5] embedded (0.20s)
  [2/5] hybrid search — 7 candidates (1.31s)
  [3/5] reranked to top 5 (1.94s)
  [4/5] fetched 2 parents (3.98s)
  [5/5] generated (18.31s)

Total: 18.31s

PaperSloth (Study assistant for UTP students).
Find and understand past year exam questions.
"pressure measurement calculate LRV URV".

        *   Clear presentation with question number and marks.
        *   State semester/year.
        *   List all relevant questions from context.
        *   No made-up questions.
        *   Concise and focused.

    *   RBB3013 | September 2025 | Q1: Precision power supply (Voltage), PMMC Voltmeter. (Not relevant)
    *   RBB3013 | September 2025 | Q2a: Pressured vessel level measurement. "Solve the Lower Range Value (LRV) and Upper Range Value (URV)." (Relevant)
    *   RBB3013 | September 2025 | Q2b: Wet leg requirement for level measurement. (Relevant to pressure/level measurement, but doesn't explicitly ask to "calculate LRV/URV").
    *   RBB3013 | September 2025 

/var/folders/my/2fkdjjvj7h9dn3nmk1ttlccc0000gn/T/ipykernel_19889/1129419027.py:20: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(_cache_key(query, filters), CACHE_TTL, json.dumps(result))


In [26]:
# ── Test 3: High marks questions only ────────────────────────────────────────
result = retrieve_and_generate(
    query       = "PI controller feedback loop",
    course_code = "RBB3013",
    min_marks   = 10,
)
print("\n" + "="*60)
print(result["answer"])

  [1/5] embedded (0.20s)
  [2/5] hybrid search — 6 candidates (1.26s)
  [3/5] reranked to top 5 (1.86s)
  [4/5] fetched 4 parents (4.01s)
  [5/5] generated (16.11s)

Total: 16.11s

PaperSloth (helpful study assistant for UTP students).
Find and understand past year exam questions.
"PI controller feedback loop".
A provided list of questions from RBB3013 | September 2025.

    *   Q1: Precision power supply (voltage measurements), PMMC multirange voltmeter. (Not related).
    *   Q2: Pressure vessel level measurement, Wet leg, Thermocouples. (Not related).
    *   Q3: Flow control loop, Pressure terminologies (psia, psig, psi), 5G systems. (Not related).
    *   Q4a: "You would like to implement a PI controller in a single feedback loop control... analyse whether a PI controller can achieve zero offset...". (This matches the query "PI controller feedback loop").

    *   Course/Semester: RBB3013 | September 2025.
    *   Question Number: Q4a.
    *   Marks: 20 marks.
    *   Content: The

/var/folders/my/2fkdjjvj7h9dn3nmk1ttlccc0000gn/T/ipykernel_19889/1129419027.py:20: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(_cache_key(query, filters), CACHE_TTL, json.dumps(result))


In [27]:
# ── Test 4: Run the same query twice — second should be cached ────────────────
print("First run:")
r1 = retrieve_and_generate(query="standard deviation voltage error analysis", course_code="RBB3013")
print()
print("Second run (should be instant cache hit):")
r2 = retrieve_and_generate(query="standard deviation voltage error analysis", course_code="RBB3013")
print(f"Cached: {r2['cached']}")

First run:
  [1/5] embedded (0.23s)
  [2/5] hybrid search — 16 candidates (1.34s)
  [3/5] reranked to top 5 (1.80s)
  [4/5] fetched 2 parents (3.87s)
  [5/5] generated (25.72s)

Total: 25.72s

Second run (should be instant cache hit):
⚡ Cache hit
Cached: True


/var/folders/my/2fkdjjvj7h9dn3nmk1ttlccc0000gn/T/ipykernel_19889/1129419027.py:20: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(_cache_key(query, filters), CACHE_TTL, json.dumps(result))


In [28]:
# ── Test 5: Inspect sources returned ─────────────────────────────────────────
# For the frontend, we need to know what each source contains
result = retrieve_and_generate(
    query       = "voltmeter resistance series shunt",
    course_code = "RBB3013",
)
print("\nANSWER:")
print(result["answer"])
print("\nSOURCES RETURNED TO FRONTEND:")
for s in result["sources"]:
    print(f"  Q{s['question_number']} | {s['semester']} {s['year']} | {s['total_marks']}m")
    print(f"  Images: {list(s['image_urls'].keys())}")
    print(f"  Text:   {s['full_text'][:100]}...")

  [1/5] embedded (0.20s)
  [2/5] hybrid search — 16 candidates (1.65s)
  [3/5] reranked to top 5 (2.10s)
  [4/5] fetched 1 parents (4.18s)
  [5/5] generated (23.38s)

Total: 23.38s

ANSWER:
*   User's Role: PaperSloth (UTP study assistant).
    *   Task: Find and present past year exam questions based on the provided context.
    *   Student Query: "voltmeter resistance series shunt".
    *   Constraints: Clear presentation (question number, marks), state semester/year, list all relevant questions, do not invent questions, concise response.

    *   Q1a.i, Q1a.ii, Q1a.iii: Focus on voltage measurements, average value, standard deviation, random errors, and error analysis. (Not related to voltmeter resistance/series shunt).
    *   Q1b.i: Focuses on a multirange voltmeter, PMMC instrument, series shunt resistances, determining $R_1, R_2, R_3, R_4$. (Matches query).
    *   Q1b.ii: Focuses on multirange voltmeter, additional shunt resistance $R_s$ for 5000V range. (Matches query).

    *

/var/folders/my/2fkdjjvj7h9dn3nmk1ttlccc0000gn/T/ipykernel_19889/1129419027.py:20: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(_cache_key(query, filters), CACHE_TTL, json.dumps(result))


In [30]:
# ── Test 1: Topic search ─────────────────────────────────────────────────────
result = retrieve_and_generate(
    query       = "give me Q1 from RBB3013 September 2025",
    course_code = "RBB3013",
)
print("\n" + "="*60)
print(result["answer"])

  [1/5] embedded (0.21s)
  [2/5] hybrid search — 16 candidates (1.27s)
  [3/5] reranked to top 5 (1.61s)
  [4/5] fetched 2 parents (3.64s)
  [5/5] generated (38.13s)

Total: 38.15s

*   User Identity: PaperSloth (UTP study assistant).
    *   Goal: Help students find/understand past year exam questions.
    *   Constraints:
        *   Clear presentation (question number and marks).
        *   State semester/year.
        *   List all relevant questions found in context.
        *   No hallucinations (if not found, say so).
        *   Concise and focused.
    *   Student Query: "give me Q1 from RBB3013 September 2025".

    *   Looking for `RBB3013 | September 2025`.
    *   Found `[RBB3013 | September 2025 2025 | Q1 | 25 marks]`.
    *   Parts of Q1: Q1a.i, Q1a.ii, Q1a.iii, Q1b.i, Q1b.ii.

    *   Header: Semester/Year.
    *   Body: List each part of Q1 with its marks.
**RBB3013 | September 2025**

**Q1 [25 marks]**

**Q1a.i:** The output voltage from a precision 12-V power supply 

/var/folders/my/2fkdjjvj7h9dn3nmk1ttlccc0000gn/T/ipykernel_19889/1129419027.py:20: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(_cache_key(query, filters), CACHE_TTL, json.dumps(result))
